In [ ]:
# ==========================================
# BLOCK 1: SMART ENVIRONMENT SETUP
# ==========================================
# @title ⚙️ 1. Setup Environment & Persistence
# @markdown ---
# @markdown ### 📂 Workspace Configuration
# @markdown Define your Google Drive workspace and optional shared drive paths below.
# @markdown ---
DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}

import os
import sys
import shutil
from datetime import datetime
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

# --- WORKSPACE PATHS ---
LOCAL_WORKSPACE = "/content/AutoScribe_Local"
TEMP_DIR = f"{LOCAL_WORKSPACE}/temp_media"
LOCAL_OUTPUT = f"{LOCAL_WORKSPACE}/outputs"
LOCAL_LOG = f"{LOCAL_WORKSPACE}/autoscribe_debug.log"
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

def log_event(level, message, print_to_console=True):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"[{timestamp}] {level}: {message}"
    with open(LOCAL_LOG, "a", encoding="utf-8") as f:
        f.write(log_entry + "\n")
    if print_to_console:
        color = {"INFO": "32", "WARNING": "33", "ERROR": "31"}.get(level, "37")
        print(f"\033[{color}m{log_entry}\033[0m", flush=True)

def inf(msg, style, wdth):
    display(widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth)))

if not os.path.exists('/content/drive/MyDrive'):
    log_event("INFO", "Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
else:
    log_event("INFO", "✅ Google Drive already mounted.")

root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}" if SHARED_DRIVE_NAME and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}") else "/content/drive/MyDrive"
workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
DRIVE_BASE = f"{root_path}/{workspace_name}"
DRIVE_OUTPUT_DIR = os.path.join(DRIVE_BASE, "outputs")
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# --- STRICT DEPENDENCY MATRIX ---
# Enforcing explicit versions to prevent upstream breaking changes from Hugging Face
log_event("INFO", "Validating and enforcing strict dependency matrix...")
with capture.capture_output() as cap:
    !pip install -q yt-dlp faster-whisper ffmpeg-python accelerate torch torchvision einops
    !pip install -q transformers==4.44.2 tokenizers==0.19.1
    !pip install -q --upgrade --no-deps yt-dlp

os.environ["HF_HOME"] = "/content/local_hf_cache"
BLOCK_1_COMPLETED = True
clear_output()
inf('\u2714 Environment Synchronized (Strict Dependency Matrix)', 'success', '400px')

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING
# ==========================================
# @title 📥 2. Source Configuration & Routing
# @markdown ---
# @markdown ### ⚙️ 1. Select Source
SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive"]

# @markdown ---
# @markdown ### 🔗 Option A: URL (YouTube)
YOUTUBE_URL = "https://www.youtube.com/playlist?list=PLvQWpZ46MVvjM2vX-pkeBxLUMgd8T9bRZ" # @param {type:"string"}

# @markdown ### 📂 Option B: Google Drive
DRIVE_FILE_PATH = "/content/drive/MyDrive/your_video.mp4" # @param {type:"string"}

# @markdown ---
# @markdown ### 🧠 2. AI Processing Settings
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}
# @markdown ---

import yt_dlp
import subprocess
import shutil
import os
import re

# Global routing queue for downstream AI processing
routing_queue = []

if 'BLOCK_1_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Block 1 must be executed prior to Block 2.")

def analyze_and_route(file_path, title, video_id):
    """Analyzes media streams to route files to appropriate AI models."""
    log_event("INFO", f"Analyzing media streams for: {title}...")
    
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "a", 
        "-show_entries", "stream=codec_type", "-of", "default=nw=1:nk=1", file_path
    ]
    
    try:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Media file not found at path: {file_path}")
            
        res = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
        # Verify if stdout actually contains the word 'audio'
        has_audio = "audio" in res.stdout.strip().lower()
        
    except Exception as e:
        log_event("WARNING", f"Stream probe failed for {title}. Defaulting to audio pipeline. Error: {e}")
        has_audio = True # Default assumption if probe fails on existing file
        
    # Determine processing path based on audio presence and user configuration
    use_vision = (WHISPER_MODE == "Off") or (WHISPER_MODE == "Auto" and not has_audio)
    
    if use_vision and not VISION_FALLBACK:
        log_event("WARNING", f"No audio detected for {title}, but Vision Fallback is disabled.")
        
    routing_queue.append({
        'id': video_id, 
        'title': title, 
        'path': file_path, 
        'use_vision': use_vision
    })
    
    route_str = "Vision Analysis" if use_vision else "Whisper Transcription"
    log_event("INFO", f"[ROUTING]: Assigned to {route_str}.")

# --- INGESTION EXECUTION ---
log_event("INFO", f"Initializing ingestion protocol. Source: {SOURCE_TYPE}")

if SOURCE_TYPE == "Google_Drive":
    if os.path.exists(DRIVE_FILE_PATH):
        fname = os.path.basename(DRIVE_FILE_PATH)
        SESSION_NAME = re.sub(r'[^\w\s-]', '', fname).strip().replace(' ', '_')[:50]
        dest = os.path.join(TEMP_DIR, fname)
        
        if not os.path.exists(dest):
            log_event("INFO", f"Transferring {fname} to local NVMe workspace...")
            shutil.copy2(DRIVE_FILE_PATH, dest)
        else:
            log_event("INFO", f"✅ Local cache hit for {fname}. Bypassing transfer.")
            
        analyze_and_route(dest, fname, fname.split('.')[0])
    else:
        log_event("ERROR", f"Target file not found in Google Drive directory: {DRIVE_FILE_PATH}")

elif SOURCE_TYPE == "URL":
    common_opts = {
        'extract_flat': 'in_playlist', 
        'quiet': True, 
        'extractor_args': {'youtube': ['player_client=android']}
    }
    
    with yt_dlp.YoutubeDL(common_opts) as ydl:
        try:
            info = ydl.extract_info(YOUTUBE_URL, download=False)
            raw_title = info.get('title', 'Media_Batch')
            SESSION_NAME = re.sub(r'[^\w\s-]', '', raw_title).strip().replace(' ', '_')[:50]
            log_event("INFO", f"Session Workspace Established: {SESSION_NAME}")
            
            entries = info['entries'] if 'entries' in info else [info]
            
            for entry in entries:
                try:
                    v_id = entry['id']
                    v_title = entry.get('title', v_id)
                    
                    dl_opts = {
                        'format': 'best', 
                        'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s', 
                        'extractor_args': {'youtube': ['player_client=android']},
                        'source_address': '0.0.0.0', 
                        'quiet': True,
                        'ignoreerrors': True
                    }
                    
                    with yt_dlp.YoutubeDL(dl_opts) as ydl_dl:
                        # BULLETPROOF PATH RESOLUTION: Find file by ID to ignore extension guessing
                        existing_files = [f for f in os.listdir(TEMP_DIR) if f.startswith(v_id + ".")]
                        
                        if existing_files:
                            actual_loc = os.path.join(TEMP_DIR, existing_files[0])
                            log_event("INFO", f"✅ Local cache hit for {v_title}.")
                        else:
                            log_event("INFO", f"Downloading media: {v_title}...")
                            dl_info = ydl_dl.extract_info(entry['url'], download=True)
                            
                            if dl_info is None:
                                raise Exception("Downloader failed to extract payload information.")
                                
                            # Capture the exact path generated by yt-dlp
                            actual_loc = ydl_dl.prepare_filename(dl_info)
                            
                        analyze_and_route(actual_loc, v_title, v_id)
                        
                except Exception as entry_ex:
                    log_event("ERROR", f"Failed to ingest entry '{v_title}'. Error: {entry_ex}")
                    continue 
                    
        except Exception as ex:
            log_event("ERROR", f"URL Ingestion pipeline encountered a critical failure: {ex}")

BLOCK_2_COMPLETED = True
log_event("INFO", f"✅ Block 2 Execution Complete. {len(routing_queue)} item(s) pending processing.")

In [ ]:
# ==========================================
# BLOCK 3: ADAPTIVE AI PROCESSING
# ==========================================
# @title 🧠 3. Execute AI Processing
# @markdown ---
# @markdown ### 🤖 Multi-Stage Adaptive Engine
# @markdown This block executes AI inference utilizing a duration-based scaling architecture:
# @markdown * **Tier 1 (< 2h):** `large-v3` (Maximum Precision)
# @markdown * **Tier 2 (2-5h):** `medium` (Balanced Performance)
# @markdown * **Tier 3 (5-8h):** `small` (Enhanced Stability)
# @markdown * **Tier 4 (> 8h):** `tiny` (Secure / Resource Optimized)
# @markdown ---

import gc
import torch
import subprocess
import os
import re
import shutil
from datetime import datetime
from PIL import Image

# Validate operational prerequisites
if 'BLOCK_2_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Preceding environment and ingestion blocks must be completed.")

# Resolve the designated output directory using the dynamically generated session name
session_folder = SESSION_NAME if 'SESSION_NAME' in locals() else datetime.now().strftime("%Y-%m-%d_%H-%M")
LOCAL_EXP = os.path.join(LOCAL_OUTPUT, session_folder)
os.makedirs(LOCAL_EXP, exist_ok=True)

try:
    from faster_whisper import WhisperModel
except ImportError:
    log_event("ERROR", "Dependency 'faster-whisper' is missing. Verify Block 1 initialization.")

def get_duration(file_path):
    """Calculates media duration in minutes via ffprobe metadata extraction."""
    cmd = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration", 
        "-of", "default=noprint_wrappers=1:nokey=1", file_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
    return float(result.stdout) / 60 if result.stdout else 0

# --- WHISPER TRANSCRIPTION PIPELINE ---
audio_tasks = [t for t in routing_queue if not t['use_vision']]

if audio_tasks:
    for task in audio_tasks:
        duration = get_duration(task['path'])
        
        # Implement adaptive model scaling logic to prevent VRAM exhaustion
        if duration < 120:
            model_size = "large-v3"
        elif duration < 300:
            model_size = "medium"
        elif duration < 480:
            model_size = "small"
        else:
            model_size = "tiny"
            
        log_event("INFO", f"🎙️ Initializing {model_size} model for {task['title']} ({duration:.1f} min)...")
        
        try:
            # Load model instance onto GPU utilizing FP16 precision
            model = WhisperModel(model_size, device="cuda", compute_type="float16", download_root=os.environ["HF_HOME"])
            
            # Execute transcription with Voice Activity Detection (VAD) filter enforced
            segments, _ = model.transcribe(
                task['path'], 
                language="sv", 
                vad_filter=True,
                condition_on_previous_text=False
            )
            
            safe_name = re.sub(r'[^\w\s-]', '', task['title']).strip().replace(' ', '_')[:50]
            output_file = os.path.join(LOCAL_EXP, f"{safe_name}.md")
            
            # Compile results into standard Markdown structure
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(f"# Source: {task['title']}\n")
                f.write(f"**Execution Timestamp:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
                f.write(f"**Pipeline Metadata:** Duration {duration:.1f}m | Core Model: {model_size}\n\n")
                f.write("## Transcription\n\n")
                for s in segments:
                    f.write(f"[{int(s.start)//60:02d}:{int(s.start)%60:02d}] {s.text.strip()}\n")
                    
            log_event("INFO", f"✅ Transcription finalized for: {task['title']}")
            
        except Exception as e:
            log_event("ERROR", f"Transcription pipeline critical failure on {task['title']}: {e}")
        finally:
            # Execute aggressive memory purge to maintain kernel stability across batch tasks
            if 'model' in locals():
                del model
            gc.collect()
            torch.cuda.empty_cache()

# --- VISION ANALYSIS PIPELINE (MOONDREAM2) ---
vision_tasks = [t for t in routing_queue if t['use_vision']]

if vision_tasks:
    log_event("INFO", "--- Initializing Moondream2 Vision Architecture ---")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        
        model_id = "vikhyatk/moondream2"
        # STRICT DEPENDENCY PINNING: Prevents remote config mismatches
        stable_revision = "2024-08-26"
        
        tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            revision=stable_revision,
            trust_remote_code=True, 
            cache_dir=os.environ["HF_HOME"]
        )
        moondream = AutoModelForCausalLM.from_pretrained(
            model_id, 
            revision=stable_revision,
            trust_remote_code=True, 
            cache_dir=os.environ["HF_HOME"]
        ).to("cuda")
        moondream.eval()

        for task in vision_tasks:
            log_event("INFO", f"👁️ Commencing visual frame analysis: {task['title']}")
            safe_name = re.sub(r'[^\w\s-]', '', task['title']).strip().replace(' ', '_')[:50]
            export_path = os.path.join(LOCAL_EXP, f"{safe_name}_vision.md")
            
            # Frame extraction directory configuration
            frame_dir = os.path.join(TEMP_DIR, f"frames_{task['id']}")
            os.makedirs(frame_dir, exist_ok=True)
            
            # Execute ffmpeg process to extract frames at 1/30 FPS
            subprocess.run([
                "ffmpeg", "-i", task['path'], "-vf", "fps=1/30", 
                f"{frame_dir}/frame_%04d.jpg", "-hide_banner", "-loglevel", "error"
            ])
            
            frames = sorted(os.listdir(frame_dir))
            
            with open(export_path, "w", encoding="utf-8") as f:
                f.write(f"# Source: {task['title']}\n")
                f.write(f"**Pipeline Metadata:** Visual Frame Analysis (Moondream2)\n\n")
                f.write("## Visual Description Log\n\n")
                
                for i, frame_file in enumerate(frames):
                    try:
                        img_path = os.path.join(frame_dir, frame_file)
                        enc_image = moondream.encode_image(Image.open(img_path))
                        answer = moondream.answer_question(
                            enc_image, 
                            "Describe the primary action, subject, or text visible in this frame concisely.", 
                            tokenizer
                        )
                        f.write(f"[{divmod(i * 30, 60)[0]:02d}:{divmod(i * 30, 60)[1]:02d}] {answer}\n")
                    except Exception as e:
                        log_event("WARNING", f"Frame extraction bypassed due to read error: {e}")
                        
    except Exception as e:
        log_event("ERROR", f"Vision model instantiation failed: {e}")
    finally:
        # Purge VLM architecture from GPU memory
        if 'moondream' in locals():
            del moondream
        gc.collect()
        torch.cuda.empty_cache()

# --- REMOTE SYNCHRONIZATION ---
log_event("INFO", f"📤 Executing directory synchronization to Google Drive: /outputs/{session_folder}")
DRIVE_DESTINATION = os.path.join(DRIVE_OUTPUT_DIR, session_folder)

try:
    # Ensure dirs_exist_ok is enforced to allow appending data to an existing session directory
    shutil.copytree(LOCAL_EXP, DRIVE_DESTINATION, dirs_exist_ok=True)
    BLOCK_3_COMPLETED = True
    log_event("INFO", "✅ Block 3 Finalized. Export payload securely synchronized to remote storage.")
except Exception as e:
    log_event("ERROR", f"Remote synchronization operation encountered an exception: {e}")

In [ ]:
# ==========================================
# BLOCK 4: FINALIZE & DISCONNECT
# ==========================================
# @title 📝 4. Finalize & Disconnect
# @markdown ---
# @markdown ### 🧹 Housekeeping & Cleanup
# @markdown **Warning:** This will export final logs, wipe the local NVMe cache, and disconnect the session.
# @markdown ---
CONFIRM_SHUTDOWN = True # @param {type:"boolean"}

from google.colab import runtime
import shutil
import os

# Path references mapped from Block 1
LOCAL_WORKSPACE = "/content/AutoScribe_Local"
LOCAL_LOG = f"{LOCAL_WORKSPACE}/autoscribe_debug.log"

if 'DRIVE_OUTPUT_DIR' not in locals():
    print("⚠️ WARNING: Workspace variables missing. Ensure Block 1 was executed.", flush=True)
elif 'BLOCK_3_COMPLETED' in locals() and CONFIRM_SHUTDOWN:
    print("🧹 Initiating final workspace cleanup...", flush=True)
    
    # --- LOG EXPORT ARCHITECTURE ---
    if os.path.exists(LOCAL_LOG):
        # Resolve dynamic session namespace
        session_folder = SESSION_NAME if 'SESSION_NAME' in locals() else "AutoScribe_Orphaned_Logs"
        
        # Define isolated logs subdirectory within the session folder
        session_target_dir = os.path.join(DRIVE_OUTPUT_DIR, session_folder)
        logs_target_dir = os.path.join(session_target_dir, "logs")
        
        # Ensure directory structures exist
        os.makedirs(logs_target_dir, exist_ok=True)
        
        # Format standardized log filename
        drive_log_path = os.path.join(logs_target_dir, f"{session_folder}_debug.log")
        
        try:
            shutil.copy2(LOCAL_LOG, drive_log_path)
            print(f"✅ Debug log securely exported to isolated directory: {drive_log_path}", flush=True)
        except Exception as e:
            print(f"❌ Failed to export log file payload: {e}", flush=True)
    else:
        print("⚠️ No local debug log detected for export.", flush=True)
    
    # --- WORKSPACE PURGE ---
    if os.path.exists(LOCAL_WORKSPACE):
        shutil.rmtree(LOCAL_WORKSPACE)
        print("✅ Local NVMe storage successfully purged.", flush=True)
    
    print("🎉 WORKFLOW COMPLETE - Terminating runtime session to preserve Compute Units...", flush=True)
    runtime.unassign()
else:
    print("⚠️ Shutdown procedure aborted or Block 3 execution incomplete.", flush=True)